In [1]:


import os
import json
import sqlite3
from pathlib import Path

import pandas as pd

# Caminho do projeto (ajuste se abrir o notebook de outro lugar)
PROJECT_ROOT = Path("..").resolve()

# Caminho do banco do Edge
DB_PATH = PROJECT_ROOT / "simulacao" / "app" / "queue.db"
DB_PATH

# Testar se o arquivo existe
if not DB_PATH.exists():
    raise FileNotFoundError(f"queue.db não encontrado em: {DB_PATH}")

conn = sqlite3.connect(DB_PATH)
conn


In [2]:


df_events = pd.read_sql_query("SELECT * FROM events", conn)
df_events.head()

,id,ts,created_at,event_json,source,source_topic,synced,synced_at
0,1,1761778511774,1761778511774,"{""event_id"": null, ""timestamp"": ""2025-10-29T22...",api,None,1,1761778623341
1,2,1761778511774,1761778511774,"{""event_id"": null, ""timestamp"": ""2025-10-29T22...",api,None,1,1761778623341
2,3,1761778511774,1761778511774,"{""event_id"": null, ""timestamp"": ""2025-10-29T22...",api,None,1,1761778623341
3,4,1761778512012,1761778512012,"{""event_id"": null, ""timestamp"": ""2025-10-29T22...",api,None,1,1761778623341
4,5,1761778512044,1761778512044,"{""event_id"": null, ""timestamp"": ""2025-10-29T22...",api,None,1,1761778623341


In [3]:

# Garantir que temos linhas
len(df_events)

# Converter cada event_json em dict
events_dicts = [json.loads(raw) for raw in df_events["event_json"]]

# Normalizar em colunas (context.*, payload.* etc)
df_json = pd.json_normalize(events_dicts, sep="__")
df_json.head()

# Juntar metadados do DB + conteúdo do JSON
cols_meta = ["id", "ts", "created_at", "source", "source_topic", "synced", "synced_at"]
df_meta = df_events[cols_meta]

df_flat = pd.concat([df_meta, df_json], axis=1)

# Ordenar colunas de forma mais legível (opcional)
ordered_cols = [
    "id", "ts", "created_at",
    "event_id", "timestamp", "totem_id", "session_id",
    "event_type", "channel",
    "context__locale", "context__modes_enabled",
    "payload__content_id", "payload__dwell_ms",
    "payload__csat", "payload__comment",
    "source", "source_topic", "synced", "synced_at",
]

df_flat = df_flat.reindex(columns=[c for c in ordered_cols if c in df_flat.columns])
df_flat.head()



,id,ts,created_at,event_id,timestamp,totem_id,session_id,event_type,channel,context__locale,context__modes_enabled,payload__content_id,payload__dwell_ms,payload__csat,payload__comment,source,source_topic,synced,synced_at
0,1,1761778511774,1761778511774,None,2025-10-29T22:55:11.473909+00:00,FM-LOCAL-01,sess-jo6nbzl7vr,consent_updated,no_touch,pt-BR,"[libras, font_xl]",NaN,NaN,NaN,NaN,api,None,1,1761778623341
1,2,1761778511774,1761778511774,None,2025-10-29T22:55:11.452268+00:00,FM-LOCAL-01,sess-p62ir9h7il,consent_updated,voice,pt-BR,"[high_contrast, libras]",NaN,NaN,NaN,NaN,api,None,1,1761778623341
2,3,1761778511774,1761778511774,None,2025-10-29T22:55:11.473550+00:00,FM-LOCAL-01,sess-wp0y7y17o7,consent_updated,voice,pt-BR,[],NaN,NaN,NaN,NaN,api,None,1,1761778623341
3,4,1761778512012,1761778512012,None,2025-10-29T22:55:12.010717+00:00,FM-LOCAL-01,sess-jo6nbzl7vr,interaction_started,no_touch,pt-BR,NaN,NaN,NaN,NaN,NaN,api,None,1,1761778623341
4,5,1761778512044,1761778512044,None,2025-10-29T22:55:12.043368+00:00,FM-LOCAL-01,sess-p62ir9h7il,interaction_started,voice,pt-BR,NaN,NaN,NaN,NaN,NaN,api,None,1,1761778623341


In [4]:

# Converter timestamp ISO → datetime (se quiser para análises)
df_flat["timestamp"] = pd.to_datetime(df_flat["timestamp"], errors="coerce")

# Campos numéricos
for col in ["payload__dwell_ms", "payload__csat"]:
    if col in df_flat.columns:
        df_flat[col] = pd.to_numeric(df_flat[col], errors="coerce")
        
df_flat.dtypes

# Converter listas em strings JSON para evitar erro no SQLite
import json

if "context__modes_enabled" in df_flat.columns:
    df_flat["context__modes_enabled"] = (
        df_flat["context__modes_enabled"]
        .apply(lambda x: json.dumps(x) if isinstance(x, list) else x)
    )



In [5]:


df_flat.to_sql("events_flat", conn, if_exists="replace", index=False)

# Conferir se criou
pd.read_sql_query("SELECT COUNT(*) AS n FROM events_flat", conn)

pd.read_sql_query("SELECT * FROM events_flat LIMIT 10", conn)



,id,ts,created_at,event_id,timestamp,totem_id,session_id,event_type,channel,context__locale,context__modes_enabled,payload__content_id,payload__dwell_ms,payload__csat,payload__comment,source,source_topic,synced,synced_at
0,1,1761778511774,1761778511774,None,2025-10-29 22:55:11.473909+00:00,FM-LOCAL-01,sess-jo6nbzl7vr,consent_updated,no_touch,pt-BR,"[""libras"", ""font_xl""]",None,None,None,None,api,None,1,1761778623341
1,2,1761778511774,1761778511774,None,2025-10-29 22:55:11.452268+00:00,FM-LOCAL-01,sess-p62ir9h7il,consent_updated,voice,pt-BR,"[""high_contrast"", ""libras""]",None,None,None,None,api,None,1,1761778623341
2,3,1761778511774,1761778511774,None,2025-10-29 22:55:11.473550+00:00,FM-LOCAL-01,sess-wp0y7y17o7,consent_updated,voice,pt-BR,[],None,None,None,None,api,None,1,1761778623341
3,4,1761778512012,1761778512012,None,2025-10-29 22:55:12.010717+00:00,FM-LOCAL-01,sess-jo6nbzl7vr,interaction_started,no_touch,pt-BR,None,None,None,None,None,api,None,1,1761778623341
4,5,1761778512044,1761778512044,None,2025-10-29 22:55:12.043368+00:00,FM-LOCAL-01,sess-p62ir9h7il,interaction_started,voice,pt-BR,None,None,None,None,None,api,None,1,1761778623341
5,6,1761778512212,1761778512212,None,2025-10-29 22:55:12.209895+00:00,FM-LOCAL-01,sess-wp0y7y17o7,interaction_started,voice,pt-BR,None,None,None,None,None,api,None,1,1761778623341
6,7,1761778512405,1761778512405,None,2025-10-29 22:55:12.402488+00:00,FM-LOCAL-01,sess-p62ir9h7il,content_selected,voice,pt-BR,None,mapa_principal,None,None,None,api,None,1,1761778623341
7,8,1761778512539,1761778512539,None,2025-10-29 22:55:12.537678+00:00,FM-LOCAL-01,sess-jo6nbzl7vr,content_selected,no_touch,pt-BR,None,rota_acessivel_banheiro,None,None,None,api,None,1,1761778623341
8,9,1761778512645,1761778512645,None,2025-10-29 22:55:12.643483+00:00,FM-LOCAL-01,sess-p62ir9h7il,content_selected,voice,pt-BR,None,ajuda_acessibilidade,None,None,None,api,None,1,1761778623341
9,10,1761778512782,1761778512782,None,2025-10-29 22:55:12.780131+00:00,FM-LOCAL-01,sess-wp0y7y17o7,content_selected,voice,pt-BR,None,ajuda_acessibilidade,None,None,None,api,None,1,1761778623341


In [6]:

df_flat_sessions = df_flat.copy()

# Garantir que temos session_id
df_flat_sessions = df_flat_sessions.dropna(subset=["session_id"])

# Helpers
def first_non_null(series):
    for v in series:
        if pd.notna(v):
            return v
    return None

group_cols = ["session_id"]

agg_dict = {
    "timestamp": ["min", "max"],                         # início/fim da sessão
    "channel": first_non_null,                          # canal principal
    "context__modes_enabled": first_non_null,           # modos de acessibilidade
    "payload__dwell_ms": "max",                         # dwell_ms final (interaction_ended)
    "payload__csat": "mean",                            # csat médio
    "id": "count",                                      # qtde de eventos na sessão
}

df_sessions = (
    df_flat_sessions
    .groupby(group_cols)
    .agg(agg_dict)
)

# Ajustar nomes de colunas
df_sessions.columns = [
    "_".join([c for c in col if c]) for col in df_sessions.columns.values
]
df_sessions = df_sessions.reset_index()

df_sessions.rename(columns={
    "timestamp_min": "session_start",
    "timestamp_max": "session_end",
    "channel_first_non_null": "channel_main",
    "context__modes_enabled_first_non_null": "modes_enabled",
    "payload__dwell_ms_max": "dwell_ms",
    "payload__csat_mean": "csat_mean",
    "id_count": "events_count",
}, inplace=True)

df_sessions.head()


,session_id,session_start,session_end,channel_main,modes_enabled,dwell_ms,csat_mean,events_count
0,sess-0acv5ik3st,2025-10-29 22:55:36.199818+00:00,2025-10-29 22:55:37.228493+00:00,touch,"[""font_xl""]",15203.0,4.0,5
1,sess-0d5qvlh6qn,2025-10-29 22:55:32.477329+00:00,2025-10-29 22:55:35.148471+00:00,touch,[],28942.0,5.0,7
2,sess-45wzwyk169,2025-10-29 22:55:17.075396+00:00,2025-10-29 22:55:20.156145+00:00,voice,"[""high_contrast""]",27828.0,NaN,6
3,sess-4h9xilxyqm,2025-10-29 22:55:17.072938+00:00,2025-10-29 22:55:20.248636+00:00,touch,"[""high_contrast"", ""libras""]",18870.0,3.0,7
4,sess-bzw8nwyamy,2025-10-29 22:55:25.693443+00:00,2025-10-29 22:55:27.055243+00:00,touch,"[""libras"", ""font_xl""]",24294.0,3.0,5


In [7]:

df_sessions.to_sql("session_metrics", conn, if_exists="replace", index=False)

pd.read_sql_query("SELECT COUNT(*) AS n FROM session_metrics", conn)


,n
0,30


In [8]:

OUTPUT_DIR = PROJECT_ROOT / "analysis" / "exports"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

df_flat.to_csv(OUTPUT_DIR / "events_flat.csv", index=False, encoding="utf-8")
df_sessions.to_csv(OUTPUT_DIR / "session_metrics.csv", index=False, encoding="utf-8")

OUTPUT_DIR


WindowsPath('C:/Users/giova/AppData/Local/Programs/Projetos/Sense-Care-Challenge/analysis/exports')